# JS6_Tugas — Perbandingan Exact NN, ANNOY, HNSW, dan FAISS

## Tujuan
Menjalankan eksperimen pencarian *nearest neighbors* pada dataset sintetis berukuran 10.000 vektor dengan 128 dimensi, lalu membandingkan waktu komputasi di:
1. **Google Colab**
2. **PyDroid3 pada smartphone Android**

Notebook ini mencatat spesifikasi runtime yang dapat dideteksi secara otomatis, waktu proses, dan kesesuaian hasil ANN terhadap Exact NN.

> **Penting untuk laporan:** saya tidak dapat menjalankan kode ini pada smartphone fisik Anda atau mengetahui spesifikasi perangkat Anda dari sini. Jalankan notebook/script di kedua perangkat, lalu gunakan hasil aktualnya. Jangan menuliskan spesifikasi atau waktu yang belum diukur seolah-olah sudah diuji.
>
> **Kompatibilitas Android:** `numpy`, `scikit-learn`, dan `scikit-learn` biasanya lebih mudah dipasang daripada library native seperti `faiss` dan `hnswlib`. Ketersediaan `annoy`, `faiss`, dan `hnswlib` bergantung pada versi Python, arsitektur Android, dan paket/wheel yang tersedia. Jika suatu library gagal diimpor pada PyDroid3, catat sebagai keterbatasan lingkungan, bukan sebagai waktu eksekusi nol.


## 1. Instalasi library

Jalankan sel ini di Google Colab. Di PyDroid3, pemasangan library native dapat gagal jika tidak tersedia paket yang kompatibel. Jika instalasi gagal, lanjutkan dengan mencatat library mana yang berhasil dipasang.

In [11]:
# Google Colab: jalankan instalasi ini jika library belum tersedia.
%pip -q install numpy scikit-learn annoy hnswlib faiss-cpu psutil


## 2. Deteksi spesifikasi runtime dan impor library

Sel ini mencatat OS, versi Python, jumlah core CPU yang terlihat, RAM total/tersedia (jika dapat dideteksi), serta status impor setiap library.

In [12]:
import os
import sys
import time
import platform
import importlib
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

try:
    import psutil
except Exception:
    psutil = None

def runtime_specs():
    info = {
        "OS/platform": platform.platform(),
        "System": platform.system(),
        "Machine/architecture": platform.machine(),
        "Python version": sys.version.split()[0],
        "Logical CPU cores": os.cpu_count(),
    }
    if psutil is not None:
        try:
            info["RAM total (GB)"] = round(psutil.virtual_memory().total / (1024**3), 2)
            info["RAM available at start (GB)"] = round(psutil.virtual_memory().available / (1024**3), 2)
        except Exception:
            pass
    return info

print("=== SPESIFIKASI RUNTIME ===")
for key, value in runtime_specs().items():
    print(f"{key}: {value}")

library_status = {}
for module_name in ["annoy", "hnswlib", "faiss"]:
    try:
        importlib.import_module(module_name)
        library_status[module_name] = "Tersedia"
    except Exception as exc:
        library_status[module_name] = f"Tidak tersedia: {type(exc).__name__}: {exc}"

print("\n=== STATUS LIBRARY ===")
for name, status in library_status.items():
    print(f"{name}: {status}")


=== SPESIFIKASI RUNTIME ===
OS/platform: Linux-6.6.122+-x86_64-with-glibc2.39
System: Linux
Machine/architecture: x86_64
Python version: 3.13.16
Logical CPU cores: 2
RAM total (GB): 12.67
RAM available at start (GB): 11.53

=== STATUS LIBRARY ===
annoy: Tersedia
hnswlib: Tersedia
faiss: Tersedia


## 3. Konfigurasi dan pembuatan dataset

Dataset memiliki 10.000 vektor dengan 128 fitur. Untuk menjaga agar eksperimen lebih memungkinkan dijalankan pada smartphone, pencarian waktu dan evaluasi dilakukan pada subset query yang dapat diubah melalui `N_QUERY`. Indeks tetap dibuat dari seluruh 10.000 vektor. Gunakan nilai yang sama di Colab dan Android untuk perbandingan yang lebih adil.

In [13]:
SEED = 42
N_SAMPLES = 10_000
DIM = 128
K = 10
N_QUERY = 1_000  # ubah menjadi N_SAMPLES untuk menguji seluruh data, jika perangkat kuat

np.random.seed(SEED)
X = np.random.random((N_SAMPLES, DIM)).astype(np.float32)

scaler = StandardScaler()
X_scaled = np.ascontiguousarray(scaler.fit_transform(X).astype(np.float32))

# Pilih query yang sama dan dapat direproduksi untuk semua metode.
query_ids = np.random.default_rng(SEED).choice(N_SAMPLES, size=min(N_QUERY, N_SAMPLES), replace=False)
X_query = np.ascontiguousarray(X_scaled[query_ids])

print("Dataset shape:", X_scaled.shape)
print("Query shape:", X_query.shape)
print("k =", K)


Dataset shape: (10000, 128)
Query shape: (1000, 128)
k = 10


## 4. Exact Nearest Neighbors (ground truth)

Exact NN menjadi acuan hasil. Untuk menghindari penggunaan memori yang sangat besar saat mencari tetangga semua query sekaligus, pencarian dijalankan dalam batch. Query itu sendiri dikeluarkan dari daftar tetangga, agar tidak selalu menjadi hasil terdekat dengan jarak nol.

In [14]:
def exact_search_batched(X, query_ids, k=10, batch_size=128):
    model = NearestNeighbors(n_neighbors=k + 1, algorithm="brute", metric="euclidean", n_jobs=-1)
    model.fit(X)
    all_ids = []
    start = time.perf_counter()
    for start_idx in range(0, len(query_ids), batch_size):
        ids_batch = query_ids[start_idx:start_idx + batch_size]
        distances, indices = model.kneighbors(X[ids_batch])
        for qid, row in zip(ids_batch, indices):
            row = [int(i) for i in row if int(i) != int(qid)]
            all_ids.append(row[:k])
    elapsed = time.perf_counter() - start
    return all_ids, elapsed

exact_ids, exact_query_seconds = exact_search_batched(
    X_scaled, query_ids, k=K, batch_size=128
)
print(f"Exact NN query time: {exact_query_seconds:.4f} seconds")
print(f"Average exact query time: {exact_query_seconds / len(query_ids) * 1000:.4f} ms/query")


Exact NN query time: 0.4099 seconds
Average exact query time: 0.4099 ms/query


In [15]:
def recall_at_k(predicted, ground_truth, k=10):
    correct = 0
    total = len(ground_truth) * k
    for pred, truth in zip(predicted, ground_truth):
        correct += len(set(pred[:k]) & set(truth[:k]))
    return correct / total if total else 0.0

def remove_query_self(pred_rows, query_ids, k=10):
    cleaned = []
    for row, qid in zip(pred_rows, query_ids):
        cleaned.append([int(i) for i in row if int(i) >= 0 and int(i) != int(qid)][:k])
    return cleaned

results = [{
    "Method": "Exact NN",
    "Build time (s)": np.nan,
    "Query time (s)": exact_query_seconds,
    "ms/query": exact_query_seconds / len(query_ids) * 1000,
    "QPS": len(query_ids) / exact_query_seconds if exact_query_seconds else np.inf,
    "Recall@10": 1.0,
    "Status": "Berhasil"
}]
predictions = {"Exact NN": exact_ids}


## 5. Benchmark ANNOY

Waktu build mencakup penambahan item dan pembangunan 10 pohon. Waktu query hanya mencakup pencarian pada query yang dipilih.

In [16]:
try:
    from annoy import AnnoyIndex

    f = X_scaled.shape[1]
    index_annoy = AnnoyIndex(f, "euclidean")

    t0 = time.perf_counter()
    for i, vec in enumerate(X_scaled):
        index_annoy.add_item(i, vec.tolist())
    index_annoy.build(10)
    annoy_build_seconds = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw_annoy = [
        index_annoy.get_nns_by_vector(q.tolist(), K + 1, search_k=-1)
        for q in X_query
    ]
    annoy_query_seconds = time.perf_counter() - t0
    annoy_ids = remove_query_self(raw_annoy, query_ids, K)
    annoy_recall = recall_at_k(annoy_ids, exact_ids, K)

    results.append({
        "Method": "ANNOY",
        "Build time (s)": annoy_build_seconds,
        "Query time (s)": annoy_query_seconds,
        "ms/query": annoy_query_seconds / len(query_ids) * 1000,
        "QPS": len(query_ids) / annoy_query_seconds if annoy_query_seconds else np.inf,
        "Recall@10": annoy_recall,
        "Status": "Berhasil"
    })
    predictions["ANNOY"] = annoy_ids
    print(f"ANNOY build: {annoy_build_seconds:.4f}s")
    print(f"ANNOY query: {annoy_query_seconds:.4f}s")
    print(f"ANNOY Recall@10: {annoy_recall:.4f}")
except Exception as exc:
    print("ANNOY tidak dapat dijalankan pada runtime ini:", repr(exc))
    results.append({
        "Method": "ANNOY", "Build time (s)": np.nan, "Query time (s)": np.nan,
        "ms/query": np.nan, "QPS": np.nan, "Recall@10": np.nan,
        "Status": f"Gagal/tidak tersedia: {type(exc).__name__}"
    })


ANNOY build: 0.1050s
ANNOY query: 0.0309s
ANNOY Recall@10: 0.0778


## 6. Benchmark HNSWLIB

`M=16` dan `ef_construction=200` digunakan untuk membangun indeks; `ef=200` digunakan saat pencarian. Parameter tersebut memengaruhi trade-off antara waktu build, waktu query, penggunaan memori, dan recall.

In [17]:
try:
    import hnswlib

    p_hnsw = hnswlib.Index(space="l2", dim=DIM)
    t0 = time.perf_counter()
    p_hnsw.init_index(max_elements=N_SAMPLES, ef_construction=200, M=16, random_seed=SEED)
    p_hnsw.add_items(X_scaled, np.arange(N_SAMPLES))
    hnsw_build_seconds = time.perf_counter() - t0

    p_hnsw.set_ef(200)
    t0 = time.perf_counter()
    labels_hnsw, _ = p_hnsw.knn_query(X_query, k=K + 1)
    hnsw_query_seconds = time.perf_counter() - t0
    hnsw_ids = remove_query_self(labels_hnsw.tolist(), query_ids, K)
    hnsw_recall = recall_at_k(hnsw_ids, exact_ids, K)

    results.append({
        "Method": "HNSWLIB",
        "Build time (s)": hnsw_build_seconds,
        "Query time (s)": hnsw_query_seconds,
        "ms/query": hnsw_query_seconds / len(query_ids) * 1000,
        "QPS": len(query_ids) / hnsw_query_seconds if hnsw_query_seconds else np.inf,
        "Recall@10": hnsw_recall,
        "Status": "Berhasil"
    })
    predictions["HNSWLIB"] = hnsw_ids
    print(f"HNSW build: {hnsw_build_seconds:.4f}s")
    print(f"HNSW query: {hnsw_query_seconds:.4f}s")
    print(f"HNSW Recall@10: {hnsw_recall:.4f}")
except Exception as exc:
    print("HNSWLIB tidak dapat dijalankan pada runtime ini:", repr(exc))
    results.append({
        "Method": "HNSWLIB", "Build time (s)": np.nan, "Query time (s)": np.nan,
        "ms/query": np.nan, "QPS": np.nan, "Recall@10": np.nan,
        "Status": f"Gagal/tidak tersedia: {type(exc).__name__}"
    })


HNSW build: 3.4236s
HNSW query: 0.3324s
HNSW Recall@10: 0.9752


## 7. Benchmark FAISS IVF

FAISS IVF menggunakan clustering untuk membatasi pencarian ke sebagian daftar vektor. Parameter `nlist=100` dan `nprobe=10` mengikuti contoh tugas. Waktu build mencakup training dan penambahan data; waktu query diukur terpisah.

In [18]:
try:
    import faiss

    quantizer = faiss.IndexFlatL2(DIM)
    index_faiss = faiss.IndexIVFFlat(quantizer, DIM, 100, faiss.METRIC_L2)

    t0 = time.perf_counter()
    index_faiss.train(X_scaled)
    index_faiss.add(X_scaled)
    faiss_build_seconds = time.perf_counter() - t0

    index_faiss.nprobe = 10
    t0 = time.perf_counter()
    _, idx_faiss = index_faiss.search(X_query, K + 1)
    faiss_query_seconds = time.perf_counter() - t0
    faiss_ids = remove_query_self(idx_faiss.tolist(), query_ids, K)
    faiss_recall = recall_at_k(faiss_ids, exact_ids, K)

    results.append({
        "Method": "FAISS IVF",
        "Build time (s)": faiss_build_seconds,
        "Query time (s)": faiss_query_seconds,
        "ms/query": faiss_query_seconds / len(query_ids) * 1000,
        "QPS": len(query_ids) / faiss_query_seconds if faiss_query_seconds else np.inf,
        "Recall@10": faiss_recall,
        "Status": "Berhasil"
    })
    predictions["FAISS IVF"] = faiss_ids
    print(f"FAISS IVF build: {faiss_build_seconds:.4f}s")
    print(f"FAISS IVF query: {faiss_query_seconds:.4f}s")
    print(f"FAISS IVF Recall@10: {faiss_recall:.4f}")
except Exception as exc:
    print("FAISS tidak dapat dijalankan pada runtime ini:", repr(exc))
    results.append({
        "Method": "FAISS IVF", "Build time (s)": np.nan, "Query time (s)": np.nan,
        "ms/query": np.nan, "QPS": np.nan, "Recall@10": np.nan,
        "Status": f"Gagal/tidak tersedia: {type(exc).__name__}"
    })


FAISS IVF build: 0.4447s
FAISS IVF query: 0.0598s
FAISS IVF Recall@10: 0.3470


## 8. Ringkasan waktu dan akurasi

Gunakan tabel ini untuk menyalin hasil Google Colab dan PyDroid3 ke laporan. Metode yang tidak berhasil diimpor akan ditandai sebagai gagal/tidak tersedia, bukan diberi waktu nol.

In [19]:
results_df = pd.DataFrame(results)
display(results_df)

print("\n=== Ringkasan Waktu (detik) ===")
for _, row in results_df.iterrows():
    if pd.notna(row["Query time (s)"]):
        print(
            f"{row['Method']:<10}: build={row['Build time (s)']:.4f}s, "
            f"query={row['Query time (s)']:.4f}s, "
            f"recall={row['Recall@10']:.4f}"
        )
    else:
        print(f"{row['Method']:<10}: {row['Status']}")


,Method,Build time (s),Query time (s),ms/query,QPS,Recall@10,Status
0,Exact NN,NaN,0.409887,0.409887,2439.699247,1.0000,Berhasil
1,ANNOY,0.104974,0.030938,0.030938,32322.225176,0.0778,Berhasil
2,HNSWLIB,3.423561,0.332372,0.332372,3008.677369,0.9752,Berhasil
3,FAISS IVF,0.444682,0.059815,0.059815,16718.108006,0.3470,Berhasil



=== Ringkasan Waktu (detik) ===
Exact NN  : build=nans, query=0.4099s, recall=1.0000
ANNOY     : build=0.1050s, query=0.0309s, recall=0.0778
HNSWLIB   : build=3.4236s, query=0.3324s, recall=0.9752
FAISS IVF : build=0.4447s, query=0.0598s, recall=0.3470


In [20]:
# Grafik: recall vs QPS untuk metode yang berhasil.
plot_df = results_df.dropna(subset=["Recall@10", "QPS"])
if not plot_df.empty:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 5))
    plt.scatter(plot_df["Recall@10"], plot_df["QPS"], s=90)
    for _, row in plot_df.iterrows():
        plt.annotate(row["Method"], (row["Recall@10"], row["QPS"]), xytext=(5, 5),
                     textcoords="offset points")
    plt.xlabel("Recall@10 (lebih tinggi lebih baik)")
    plt.ylabel("QPS (lebih tinggi lebih cepat)")
    plt.title("JS6 Tugas — Recall vs QPS")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada hasil yang dapat digrafikkan.")


## 9. Analisis hasil

Isi analisis berikut berdasarkan output nyata di masing-masing perangkat.

### Perbandingan waktu
- **Exact NN** memeriksa seluruh kandidat dan berfungsi sebagai acuan. Pada jumlah query besar, pencarian brute-force umumnya membutuhkan waktu lebih banyak.
- **ANNOY** membangun random projection trees. Waktu total di notebook dipisahkan menjadi build time dan query time. Kecepatan pencarian dipengaruhi jumlah tree dan parameter pencarian.
- **HNSWLIB** membangun graf bertingkat. Pembangunan graf bisa memerlukan biaya awal yang cukup besar, tetapi pencarian query sering kali cepat. `ef` memengaruhi keseimbangan recall dan kecepatan.
- **FAISS IVF** melatih centroid dan membagi vektor ke inverted lists. `nprobe` yang lebih besar memeriksa lebih banyak cluster dan umumnya meningkatkan recall dengan tambahan waktu query.

### Perbandingan akurasi
`Recall@10` menunjukkan proporsi tetangga exact yang ditemukan kembali oleh setiap metode ANN. Nilai mendekati 1 berarti hasilnya lebih dekat dengan Exact NN. Karena metode ANN bersifat approximate, hasil indeksnya tidak harus sama persis dengan hasil exact.

### Google Colab vs PyDroid3
Perbedaan hasil dapat disebabkan oleh:
1. CPU dan jumlah core yang tersedia.
2. Kapasitas RAM dan tekanan memori.
3. Versi Python, NumPy, scikit-learn, dan library ANN.
4. Optimasi native/BLAS dan jumlah thread.
5. Beban sistem, suhu perangkat, serta throttling termal pada smartphone.
6. Perbedaan library yang berhasil dipasang. Jika FAISS atau HNSWLIB tidak kompatibel dengan instalasi PyDroid3, perbandingan empat metode tidak dapat dilakukan secara penuh pada perangkat itu.

### Batasan penting
- Kode ini mengukur query pada `N_QUERY` sampel, bukan seluruh 10.000 data secara default. Gunakan nilai `N_QUERY` yang sama di dua lingkungan.
- Waktu build tidak sama dengan waktu query. Bandingkan keduanya secara terpisah.
- Untuk perbandingan lebih andal, jalankan eksperimen beberapa kali dan gunakan median; pengukuran pertama bisa dipengaruhi warm-up.
- Tuliskan model smartphone, RAM, versi Android, versi PyDroid3/Python, serta status library berdasarkan perangkat yang benar-benar digunakan.


## 10. Template spesifikasi perangkat untuk laporan

Lengkapi bagian Android dari menu **Settings → About phone** dan output runtime di atas. Jangan mengarang spesifikasi.

| Komponen | Google Colab | Smartphone Android / PyDroid3 |
|---|---|---|
| Model perangkat | Runtime cloud Colab | Isi model HP aktual |
| CPU / arsitektur | Output runtime specs | Output runtime specs / spesifikasi perangkat |
| RAM | Output runtime specs | RAM fisik HP dan RAM tersedia |
| OS | Linux runtime Colab | Versi Android |
| Python | Output runtime specs | Output runtime specs |
| Library tersedia | Tabel status library | Tabel status library |
| N_QUERY | Nilai yang digunakan | Gunakan nilai yang sama |
| Exact NN time | Isi hasil aktual | Isi hasil aktual |
| ANNOY build/query | Isi hasil aktual | Isi hasil aktual atau tidak tersedia |
| HNSW build/query | Isi hasil aktual | Isi hasil aktual atau tidak tersedia |
| FAISS build/query | Isi hasil aktual | Isi hasil aktual atau tidak tersedia |

## Kesimpulan
Tuliskan metode tercepat dan metode dengan recall tertinggi berdasarkan tabel hasil yang benar-benar diperoleh. Jika library native tidak dapat dipasang pada PyDroid3, jelaskan hal tersebut sebagai kendala kompatibilitas. Jangan menyimpulkan smartphone lebih lambat atau lebih cepat hanya dari dugaan; gunakan pengukuran aktual dengan dataset, jumlah query, dan parameter yang sama.
